## Imports

In [3]:
import pandas as pd
import numpy as np
import scipy
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

### Leitura do CSV baixado do Kaggle

In [4]:
caminho_arquivo = "AI_Impact_Student_Life_2026.csv"

df_completo = pd.read_csv(
    caminho_arquivo,
    sep=",",
    encoding="utf-8",
    decimal="."
)

df_completo.columns = df_completo.columns.str.strip()

### Seleção apenas das colunas de interesse

In [5]:
colunas_interesse = [
    "Student_ID",
    "Task_Frequency_Daily",
    "Main_Usage_Case",
    "GPA_Baseline",
    "GPA_Post_AI"
]

df = df_completo[colunas_interesse].copy()

### Inspeção inicial

In [6]:
print("Formato do DataFrame (linhas, colunas):", df.shape)
print("\nPrimeiras linhas:")
print(df.head())

print("\nTipos de dados por coluna:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nResumo estatístico (colunas numéricas):")
print(df.describe())

# ---- Identificação de dados duplicados ----

# Duplicatas exatas (todas as colunas iguais)
qtd_duplicadas_exatas = df.duplicated().sum()
print(f"\nQuantidade de linhas totalmente duplicadas: {qtd_duplicadas_exatas}")

# Duplicatas por Student_ID (mesmo aluno aparecendo mais de uma vez)
qtd_ids_duplicados = df.duplicated(subset="Student_ID").sum()
print(f"Quantidade de Student_ID duplicados: {qtd_ids_duplicados}")

if qtd_ids_duplicados > 0:
    print("\nExemplos de registros com Student_ID duplicado:")
    print(df[df.duplicated(subset="Student_ID", keep=False)].sort_values(by="Student_ID").head(10))

# ---- Tratamento: remoção das duplicatas por Student_ID ----
df = df.drop_duplicates(subset="Student_ID", keep="first")
print(f"\nFormato do DataFrame após remoção de duplicatas por Student_ID: {df.shape}")

Formato do DataFrame (linhas, colunas): (1500, 5)

Primeiras linhas:
  Student_ID  Task_Frequency_Daily    Main_Usage_Case  GPA_Baseline  \
0   STU-6019                     1     Code Debugging          2.62   
1   STU-6962                     3     Essay Drafting          3.99   
2   STU-2338                     2  Literature Review          2.57   
3   STU-1380                     5     Essay Drafting          2.67   
4   STU-1837                    10     Code Debugging          3.65   

   GPA_Post_AI  
0         2.62  
1         4.00  
2         2.57  
3         2.87  
4         3.85  

Tipos de dados por coluna:
Student_ID                  str
Task_Frequency_Daily      int64
Main_Usage_Case             str
GPA_Baseline            float64
GPA_Post_AI             float64
dtype: object

Valores ausentes por coluna:
Student_ID              0
Task_Frequency_Daily    0
Main_Usage_Case         0
GPA_Baseline            0
GPA_Post_AI             0
dtype: int64

Resumo estatístico (coluna

### Ajustes de formatação

In [7]:
# Garante que as colunas de GPA são numéricas
df["GPA_Baseline"] = pd.to_numeric(df["GPA_Baseline"], errors="coerce")
df["GPA_Post_AI"] = pd.to_numeric(df["GPA_Post_AI"], errors="coerce")

# Garante que a frequência de uso também é numérica
df["Task_Frequency_Daily"] = pd.to_numeric(df["Task_Frequency_Daily"], errors="coerce")

# Remove linhas com dados faltando nas colunas essenciais
df = df.dropna(subset=["GPA_Baseline", "GPA_Post_AI"])

# Remove linhas totalmente vazias
df = df.dropna(how="all")

### Coluna calculada: variação de GPA

In [8]:
df["GPA_Variacao"] = df["GPA_Post_AI"] - df["GPA_Baseline"]

### Correlação de Pearson: Task_Frequency_Daily (Frequência diária do uso de agentes IA) x GPA_Variacao

In [9]:
coef_pearson, p_valor_pearson = stats.pearsonr(df["Task_Frequency_Daily"], df["GPA_Variacao"])

print(f"\nCoeficiente de correlação de Pearson: {coef_pearson:.4f}")
print(f"P-valor (Pearson): {p_valor_pearson:.5f}")


Coeficiente de correlação de Pearson: 0.0300
P-valor (Pearson): 0.24611


### Correlação de Spearman: Task_Frequency_Daily (Frequência diária do uso de agentes IA) x GPA_Variacao

In [10]:
coef_spearman, p_valor_spearman = stats.spearmanr(df["Task_Frequency_Daily"], df["GPA_Variacao"])

print(f"\nCoeficiente de correlação de Spearman: {coef_spearman:.4f}")
print(f"P-valor (Spearman): {p_valor_spearman:.5f}")


Coeficiente de correlação de Spearman: 0.0315
P-valor (Spearman): 0.22322


### Interpretação da força da correlação

In [11]:
def interpretar_correlacao(r):
    r_abs = abs(r)
    if r_abs < 0.1:
        return "praticamente nenhuma correlação"
    elif r_abs < 0.3:
        return "correlação fraca"
    elif r_abs < 0.5:
        return "correlação moderada"
    elif r_abs < 0.7:
        return "correlação forte"
    else:
        return "correlação muito forte"

def resumo_correlacao(nome, r, p):
    direcao = "positiva" if r > 0 else "negativa"
    forca = interpretar_correlacao(r)
    significativo = "estatisticamente significativa" if p < 0.05 else "não estatisticamente significativa (p >= 0.05)"
    print(f"\n[{nome}] correlação {forca} e {direcao} (r = {r:.4f}). Resultado {significativo} (p = {p:.5f}).")

resumo_correlacao("Pearson", coef_pearson, p_valor_pearson)
resumo_correlacao("Spearman", coef_spearman, p_valor_spearman)


[Pearson] correlação praticamente nenhuma correlação e positiva (r = 0.0300). Resultado não estatisticamente significativa (p >= 0.05) (p = 0.24611).

[Spearman] correlação praticamente nenhuma correlação e positiva (r = 0.0315). Resultado não estatisticamente significativa (p >= 0.05) (p = 0.22322).


### Calculando Quartis da variação de GPA

In [12]:
gpa_var_ordenada = df["GPA_Variacao"].sort_values().reset_index(drop=True)

q1 = gpa_var_ordenada.quantile(0.25)
mediana = gpa_var_ordenada.median()
q3 = gpa_var_ordenada.quantile(0.75)
iqr = q3 - q1

print(f"Quantidade de observações: {len(gpa_var_ordenada)}")
print(f"1º quartil (Q1): {q1:.2f}")
print(f"Mediana (Q2): {mediana:.2f}")
print(f"3º quartil (Q3): {q3:.2f}")
print(f"Amplitude interquartil (IQR): {iqr:.2f}")
print(f"Valor mínimo: {gpa_var_ordenada.min():.2f} | Valor máximo: {gpa_var_ordenada.max():.2f}")

Quantidade de observações: 1500
1º quartil (Q1): 0.00
Mediana (Q2): 0.10
3º quartil (Q3): 0.20
Amplitude interquartil (IQR): 0.20
Valor mínimo: -0.10 | Valor máximo: 0.30


### Boxplot da variação de GPA

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(x=df["GPA_Variacao"], ax=ax, color="#9ecae1", width=0.45)

for valor, rotulo in [(q1, "Q1"), (mediana, "Mediana"), (q3, "Q3")]:
    ax.annotate(f"{rotulo} = {valor:.2f}", xy=(valor, 0.225), xytext=(valor, 0.33),
                ha="center", fontsize=10,
                arrowprops=dict(arrowstyle="-", color="gray"))

ax.set_title("Boxplot da variação de GPA (GPA_Post_AI − GPA_Baseline)")
ax.set_xlabel("Variação de GPA")
plt.tight_layout()
plt.show()